valid_x,yを用いて、transactionから候補リストを作成する

In [1]:
import pandas as pd

transactions = pd.read_csv(
    "C:/kaggle_nvidia/kaggle_hm/competitions/hm/data/processed/transactions_with_week.csv",
    dtype={
        "article_id": "int32",
        "week": "int8",
        "price": "float32",
        "sales_channel_id": "int8"
    }
)

valid_x = pd.read_csv(
    "C:/kaggle_nvidia/kaggle_hm/competitions/hm/data/processed/valid_x.csv",
    dtype={
        "article_id": "int32",
        "week": "int8",
        "price": "float32",
        "sales_channel_id": "int8"
    }
)

valid_y = pd.read_csv(
    "C:/kaggle_nvidia/kaggle_hm/competitions/hm/data/processed/valid_y.csv",
    dtype={
        "article_id": "int32",
        "week": "int8",
        "price": "float32",
        "sales_channel_id": "int8"
    }
)

print("読み込み完了")
print(f"transactions: {len(transactions)}行")
print(f"valid_x: {len(valid_x)}行")
print(f"valid_y: {len(valid_y)}行")

読み込み完了
transactions: 31788324行
valid_x: 31575041行
valid_y: 213283行


In [3]:
valid_x.head()

,t_dat,customer_id,article_id,price,sales_channel_id,week
0,2018-09-20,3,663713001,0.050831,2,0
1,2018-09-20,3,541518023,0.030492,2,0
2,2018-09-20,8,505221004,0.015237,2,0
3,2018-09-20,8,685687003,0.016932,2,0
4,2018-09-20,8,685687004,0.016932,2,0


先週の上位30件を候補にする

In [11]:
last_week = valid_x["week"].max()
print(f"valid_xの最終週: {last_week}")

top50 = (
    valid_x[valid_x["week"] == last_week]
    .groupby("article_id")["customer_id"]
    
    .count()
    .reset_index()
    .rename(columns={"customer_id": "count"})
    .sort_values("count", ascending=False)
    .head(50)
)

print(top50)

valid_xの最終週: 103
       article_id  count
16715   909370001   1310
17692   924243001    817
12136   865799006    750
17368   918522001    666
448     448509014    577
4728    751471001    576
17674   923758001    548
7561    809238001    539
5267    762846027    538
17354   918292001    527
7562    809238005    490
17369   918525001    433
17958   929165002    431
17693   924243002    430
17189   915529003    430
2471    673677002    420
17186   915526001    420
16865   910601003    408
3319    706016001    408
17190   915529005    408
10524   850917001    407
11874   863583001    391
15545   896169005    391
3627    715624001    390
17428   919273002    385
16716   909371001    383
7263    805947001    382
4744    751471043    379
11899   863646001    376
17120   914805002    355
3320    706016002    335
3598    714790020    334
3321    706016003    330
31      158340001    329
11888   863595006    328
10694   852584001    323
16664   908799002    320
15432   894780001    318
15776   

In [14]:
# valid_xに登場する全ユーザーを取得
all_users = valid_x["customer_id"].unique()
print(f"ユーザー数: {len(all_users)}")

# 全ユーザー × top30商品の候補ペアを作成
candidate_top50 = pd.DataFrame({
    "customer_id": all_users.repeat(50),
    "article_id": list(top50["article_id"].values) * len(all_users)
})

print(f"候補ペア数: {len(candidate_top50)}")
print(candidate_top50.head(10))

ユーザー数: 1357359
候補ペア数: 67867950
   customer_id  article_id
0            3   909370001
1            3   924243001
2            3   865799006
3            3   918522001
4            3   448509014
5            3   751471001
6            3   923758001
7            3   809238001
8            3   762846027
9            3   918292001


In [15]:
# valid_yと照合してRecallを計算する
# valid_yは「実際に買われた（ユーザー, 商品）ペア」= 正解

# 候補ペアと正解ペアをマージして一致するものを探す
hit = candidate_top50.merge(
    valid_y[["customer_id", "article_id"]],
    on=["customer_id", "article_id"],
    how="inner"
)

# Recall = 正解のうち候補に含まれた割合
recall = len(hit) / len(valid_y)
print(f"正解数: {len(valid_y)}")
print(f"候補に含まれた正解数: {len(hit)}")
print(f"Recall: {recall:.4f}")

正解数: 213283
候補に含まれた正解数: 14322
Recall: 0.0672


In [17]:
import numpy as np

def map_at_12(candidate_df, valid_y, k=12):
    """
    candidate_df: customer_id, article_id, rank列を持つDataFrame
    valid_y: customer_id, article_id列を持つDataFrame（正解）
    """
    # 正解をユーザーごとにセットにまとめる
    ground_truth = valid_y.groupby("customer_id")["article_id"].apply(set).to_dict()

    # 候補を上位k件に絞る
    top_k = candidate_df[candidate_df["rank"] <= k].sort_values(["customer_id", "rank"])

    ap_list = []
    for customer_id, group in top_k.groupby("customer_id"):
        actual = ground_truth.get(customer_id, set())
        if not actual:
            continue
        
        recommended = group["article_id"].tolist()
        hits = 0
        score = 0.0
        for i, article in enumerate(recommended, 1):
            if article in actual:
                hits += 1
                score += hits / i
        ap_list.append(score / min(len(actual), k))

    return np.mean(ap_list)

# 候補にrankを付与（top50の人気順）
candidate_top50_ranked = candidate_top50.copy()
rank_map = {article_id: rank+1 for rank, article_id in enumerate(top50["article_id"].values)}
candidate_top50_ranked["rank"] = candidate_top50_ranked["article_id"].map(rank_map)

# MAP@12を計算
map12 = map_at_12(candidate_top50_ranked, valid_y)
print(f"Recall: {recall:.4f}")
print(f"MAP@12: {map12:.4f}")

Recall: 0.0672
MAP@12: 0.0091


In [18]:
import numpy as np
# 人気上位12件に絞る
candidate_top12 = pd.DataFrame({
    "customer_id": np.repeat(all_users, 12),
    "article_id": np.tile(top50["article_id"].values[:12], len(all_users))
})

# rankを付与
candidate_top12["rank"] = np.tile(np.arange(1, 13), len(all_users))

# Recallを計算
hit = candidate_top12.merge(valid_y[["customer_id", "article_id"]], on=["customer_id", "article_id"], how="inner")
recall_baseline = len(hit) / len(valid_y)

# MAP@12を計算
map12_baseline = map_at_12(candidate_top12, valid_y)

print(f"【ベースライン】")
print(f"Recall: {recall_baseline:.4f}")
print(f"MAP@12: {map12_baseline:.4f}")

【ベースライン】
Recall: 0.0212
MAP@12: 0.0091


In [19]:
import numpy as np
import pandas as pd

# 直近1ヶ月（4週間）の購買履歴を取得
max_week_vx = valid_x["week"].max()
recent_purchase = (
    valid_x[valid_x["week"] >= max_week_vx - 3][["customer_id", "article_id", "week"]]
    .drop_duplicates(subset=["customer_id", "article_id"])
    .sort_values(["customer_id", "week"], ascending=[True, False])
)

# ユーザーごとに購買履歴にランクを付与
recent_purchase["rank"] = recent_purchase.groupby("customer_id").cumcount() + 1

# 12件以上は上位12件に絞る
recent_purchase = recent_purchase[recent_purchase["rank"] <= 12]

# 直近1ヶ月の人気ランキングを作成
popularity = (
    valid_x[valid_x["week"] >= max_week_vx - 3]
    .groupby("article_id")["customer_id"]
    .count()
    .reset_index()
    .rename(columns={"customer_id": "count"})
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
popularity["pop_rank"] = popularity.index + 1

# 全ユーザーの購買件数を取得
all_users = valid_x["customer_id"].unique()
user_purchase_count = (
    recent_purchase.groupby("customer_id")["rank"].max()
    .reindex(all_users, fill_value=0)
    .reset_index()
    .rename(columns={"rank": "purchase_count"})
)

# 人気商品上位12件を全ユーザーに展開
top12_pop = popularity.head(12)["article_id"].values
pop_all = pd.DataFrame({
    "customer_id": np.repeat(all_users, 12),
    "article_id": np.tile(top12_pop, len(all_users)),
    "pop_rank": np.tile(np.arange(1, 13), len(all_users))
})

# 購買履歴にある商品を人気候補から除外
purchased_set = recent_purchase[["customer_id", "article_id"]].copy()
purchased_set["is_purchased"] = True
pop_all = pop_all.merge(purchased_set, on=["customer_id", "article_id"], how="left")
pop_all = pop_all[pop_all["is_purchased"].isna()].drop(columns=["is_purchased"])

# ユーザーごとの購買件数を付与して残り枠を計算
pop_all = pop_all.merge(user_purchase_count, on="customer_id", how="left")
pop_all["rank"] = pop_all["pop_rank"] + pop_all["purchase_count"]
pop_all = pop_all[pop_all["rank"] <= 12][["customer_id", "article_id", "rank"]]

# 購買履歴候補と人気候補を結合
final_candidates = pd.concat([
    recent_purchase[["customer_id", "article_id", "rank"]],
    pop_all
], ignore_index=True).sort_values(["customer_id", "rank"])

print(f"候補ペア数: {len(final_candidates)}")

# Recallを計算
hit = final_candidates.merge(valid_y[["customer_id", "article_id"]], on=["customer_id", "article_id"], how="inner")
recall_new = len(hit) / len(valid_y)

# MAP@12を計算
map12_new = map_at_12(final_candidates, valid_y)

print(f"\n【新しい候補生成】")
print(f"Recall:  {recall_new:.4f}")
print(f"MAP@12:  {map12_new:.4f}")
print(f"\n【ベースライン】")
print(f"Recall:  {recall_baseline:.4f}")
print(f"MAP@12:  {map12_baseline:.4f}")

候補ペア数: 16276619

【新しい候補生成】
Recall:  0.0374
MAP@12:  0.0261

【ベースライン】
Recall:  0.0212
MAP@12:  0.0091


In [21]:
from pathlib import Path
# 全ユーザーを取得（customers.csvから）
# DATA_DIR = Path.cwd() / '../../data'
customers = pd.read_csv( 'C:\kaggle_nvidia\kaggle_hm\competitions\hm\data\processed\customers_converted.csv', usecols=['customer_id'])

# final_candidatesを提出形式に変換
submission = (
    final_candidates.sort_values(["customer_id", "rank"])
    .groupby("customer_id")["article_id"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
    .rename(columns={"article_id": "prediction"})
)

# 購買履歴のないユーザーには人気上位12件を割り当て
top12_str = " ".join(top12_pop.astype(str))
submission_full = customers.merge(submission, on="customer_id", how="left")
submission_full["prediction"] = submission_full["prediction"].fillna(top12_str)

# 保存
# OUTPUT_DIR = (Path.cwd() /  'processed').resolve()
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
submission_full.to_csv('C:\kaggle_nvidia\kaggle_hm\competitions\hm\data\processed\sub_repeat_popularity.csv', index=False)

print(f"提出ファイル保存完了")
print(f"ユーザー数: {len(submission_full)}")
print(submission_full.head(5))

<>:4: SyntaxWarning: invalid escape sequence '\k'
<>:23: SyntaxWarning: invalid escape sequence '\k'
<>:4: SyntaxWarning: invalid escape sequence '\k'
<>:23: SyntaxWarning: invalid escape sequence '\k'
C:\Users\hayat\AppData\Local\Temp\ipykernel_16852\960728151.py:4: SyntaxWarning: invalid escape sequence '\k'
  customers = pd.read_csv( 'C:\kaggle_nvidia\kaggle_hm\competitions\hm\data\processed\customers_converted.csv', usecols=['customer_id'])
C:\Users\hayat\AppData\Local\Temp\ipykernel_16852\960728151.py:23: SyntaxWarning: invalid escape sequence '\k'
  submission_full.to_csv('C:\kaggle_nvidia\kaggle_hm\competitions\hm\data\processed\sub_repeat_popularity.csv', index=False)


提出ファイル保存完了
ユーザー数: 1371980
   customer_id                                         prediction
0            1  568601043 751471001 706016001 915526001 918292...
1            2  751471001 706016001 915526001 918292001 751471...
2            3  794321007 751471001 706016001 915526001 918292...
3            4  751471001 706016001 915526001 918292001 751471...
4            5  751471001 706016001 915526001 918292001 751471...


In [22]:
print(final_candidates.columns.tolist())
print(final_candidates.head(3))
print(f"行数: {len(final_candidates)}")

['customer_id', 'article_id', 'rank']
         customer_id  article_id  rank
0                  1   568601043     1
7174107            1   751471001     2
7174108            1   706016001     3
行数: 16276619


In [24]:
import pandas as pd
from pathlib import Path

DATA_DIR = (Path.cwd() / '../../data').resolve()

articles = pd.read_csv(DATA_DIR / 'raw/articles.csv')

print(articles.columns.tolist())

['article_id', 'product_code', 'prod_name', 'product_type_no', 'product_type_name', 'product_group_name', 'graphical_appearance_no', 'graphical_appearance_name', 'colour_group_code', 'colour_group_name', 'perceived_colour_value_id', 'perceived_colour_value_name', 'perceived_colour_master_id', 'perceived_colour_master_name', 'department_no', 'department_name', 'index_code', 'index_name', 'index_group_no', 'index_group_name', 'section_no', 'section_name', 'garment_group_no', 'garment_group_name', 'detail_desc']


In [25]:
print(valid_x.columns.tolist())
print(articles.columns.tolist())

['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id', 'week']
['article_id', 'product_code', 'prod_name', 'product_type_no', 'product_type_name', 'product_group_name', 'graphical_appearance_no', 'graphical_appearance_name', 'colour_group_code', 'colour_group_name', 'perceived_colour_value_id', 'perceived_colour_value_name', 'perceived_colour_master_id', 'perceived_colour_master_name', 'department_no', 'department_name', 'index_code', 'index_name', 'index_group_no', 'index_group_name', 'section_no', 'section_name', 'garment_group_no', 'garment_group_name', 'detail_desc']


In [26]:
import os
print(os.listdir(DATA_DIR / 'processed'))

['customers_converted.csv', 'customer_id_map.csv', 'postal_code_map.csv', 'sub_baseline.csv', 'sub_latesttop12.csv', 'sub_repeat_popularity.csv', 'transactions_converted.csv', 'transactions_with_week.csv', 'valid_x.csv', 'valid_y.csv']


In [27]:
transactions = pd.read_csv(DATA_DIR / 'processed/transactions_with_week.csv', 
                           usecols=['week'], dtype={'week': 'int8'})
print(f"最小week: {transactions['week'].min()}")
print(f"最大week: {transactions['week'].max()}")
print(f"week一覧: {sorted(transactions['week'].unique())}")

最小week: 0
最大week: 104
week一覧: [np.int8(0), np.int8(1), np.int8(2), np.int8(3), np.int8(4), np.int8(5), np.int8(6), np.int8(7), np.int8(8), np.int8(9), np.int8(10), np.int8(11), np.int8(12), np.int8(13), np.int8(14), np.int8(15), np.int8(16), np.int8(17), np.int8(18), np.int8(19), np.int8(20), np.int8(21), np.int8(22), np.int8(23), np.int8(24), np.int8(25), np.int8(26), np.int8(27), np.int8(28), np.int8(29), np.int8(30), np.int8(31), np.int8(32), np.int8(33), np.int8(34), np.int8(35), np.int8(36), np.int8(37), np.int8(38), np.int8(39), np.int8(40), np.int8(41), np.int8(42), np.int8(43), np.int8(44), np.int8(45), np.int8(46), np.int8(47), np.int8(48), np.int8(49), np.int8(50), np.int8(51), np.int8(52), np.int8(53), np.int8(54), np.int8(55), np.int8(56), np.int8(57), np.int8(58), np.int8(59), np.int8(60), np.int8(61), np.int8(62), np.int8(63), np.int8(64), np.int8(65), np.int8(66), np.int8(67), np.int8(68), np.int8(69), np.int8(70), np.int8(71), np.int8(72), np.int8(73), np.int8(74), np.i

In [28]:
valid_x = pd.read_csv(DATA_DIR / 'processed/valid_x.csv', dtype={"week": "int8"})
valid_y = pd.read_csv(DATA_DIR / 'processed/valid_y.csv', dtype={"week": "int8"})

print(f"valid_x week範囲: {valid_x['week'].min()} 〜 {valid_x['week'].max()}")
print(f"valid_y week範囲: {valid_y['week'].min()} 〜 {valid_y['week'].max()}")

valid_x week範囲: 0 〜 103
valid_y week範囲: 104 〜 104


In [29]:
dtype_config = {
    "article_id": "string",
    "week": "int8",
    "price": "float32",
    "sales_channel_id": "int8"
}

transactions = pd.read_csv(DATA_DIR / 'processed/transactions_with_week.csv', dtype=dtype_config)

# 3fold分のtrain_x / train_y を作成
for fold, target_week in enumerate([103, 102, 101], 1):
    train_x = transactions[transactions["week"] < target_week]
    train_y = transactions[transactions["week"] == target_week]
    
    train_x.to_csv(DATA_DIR / f'processed/train_x_fold{fold}.csv', index=False)
    train_y.to_csv(DATA_DIR / f'processed/train_y_fold{fold}.csv', index=False)
    
    print(f"fold{fold}: train_x week {train_x['week'].min()}〜{train_x['week'].max()}, train_y week {target_week}")
    print(f"  train_x: {len(train_x)}行, train_y: {len(train_y)}行")

fold1: train_x week 0〜102, train_y week 103
  train_x: 31334398行, train_y: 240643行
fold2: train_x week 0〜101, train_y week 102
  train_x: 31060837行, train_y: 273561行
fold3: train_x week 0〜100, train_y week 101
  train_x: 30790961行, train_y: 269876行


In [30]:
def generate_candidates(df_x, n_weeks_recent=4, n_candidates=12):
    """
    候補生成関数
    df_x: 購買履歴
    n_weeks_recent: 直近何週間を使うか
    n_candidates: 候補数
    """
    max_week = df_x["week"].max()
    
    # 直近1ヶ月の購買履歴
    recent = df_x[df_x["week"] >= max_week - (n_weeks_recent - 1)]
    
    # 人気ランキング
    popularity = (
        recent.groupby("article_id")["customer_id"]
        .count()
        .reset_index()
        .rename(columns={"customer_id": "count"})
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )
    top_n_pop = popularity.head(n_candidates)["article_id"].values

    # リピート購買候補
    repeat_candidates = (
        recent[["customer_id", "article_id"]]
        .drop_duplicates()
    )
    repeat_candidates["rank"] = repeat_candidates.groupby("customer_id").cumcount() + 1
    repeat_candidates = repeat_candidates[repeat_candidates["rank"] <= n_candidates]

    # 全ユーザー
    all_users = df_x["customer_id"].unique()
    user_purchase_count = (
        repeat_candidates.groupby("customer_id")["rank"].max()
        .reindex(all_users, fill_value=0)
        .reset_index()
        .rename(columns={"rank": "purchase_count"})
    )

    # 人気商品で補完
    pop_all = pd.DataFrame({
        "customer_id": np.repeat(all_users, n_candidates),
        "article_id": np.tile(top_n_pop, len(all_users)),
        "pop_rank": np.tile(np.arange(1, n_candidates + 1), len(all_users))
    })
    purchased_set = repeat_candidates[["customer_id", "article_id"]].copy()
    purchased_set["is_purchased"] = True
    pop_all = pop_all.merge(purchased_set, on=["customer_id", "article_id"], how="left")
    pop_all = pop_all[pop_all["is_purchased"].isna()].drop(columns=["is_purchased"])
    pop_all = pop_all.merge(user_purchase_count, on="customer_id", how="left")
    pop_all["rank"] = pop_all["pop_rank"] + pop_all["purchase_count"]
    pop_all = pop_all[pop_all["rank"] <= n_candidates][["customer_id", "article_id", "rank"]]

    # 結合
    candidates = pd.concat([
        repeat_candidates[["customer_id", "article_id", "rank"]],
        pop_all
    ], ignore_index=True).sort_values(["customer_id", "rank"])

    return candidates

# 各foldで候補生成
for fold, target_week in enumerate([103, 102, 101], 1):
    print(f"\nfold{fold}の候補生成中...")
    train_x_fold = pd.read_csv(DATA_DIR / f'processed/train_x_fold{fold}.csv', dtype=dtype_config)
    candidates = generate_candidates(train_x_fold)
    candidates.to_csv(DATA_DIR / f'processed/candidates_fold{fold}.csv', index=False)
    print(f"fold{fold}: 候補ペア数 {len(candidates):,}")


fold1の候補生成中...
fold1: 候補ペア数 16,214,136

fold2の候補生成中...
fold2: 候補ペア数 16,146,179

fold3の候補生成中...
fold3: 候補ペア数 16,079,268


In [33]:
train_x_fold = pd.read_csv(DATA_DIR / f'processed/train_x_fold1.csv', dtype=dtype_config)
print(f"train_x_fold article_id型: {train_x_fold['article_id'].dtype}")
print(f"articles article_id型: {articles['article_id'].dtype}")

train_x_fold article_id型: string
articles article_id型: int64


In [34]:
articles = pd.read_csv(DATA_DIR / 'raw/articles.csv', dtype={"article_id": "string"})
print(f"articles article_id型: {articles['article_id'].dtype}")

articles article_id型: string


In [35]:
def create_features(candidates, df_x, articles):
    """
    特徴量作成関数
    candidates: 候補ペア (customer_id, article_id, rank)
    df_x: 購買履歴
    articles: 商品マスタ
    """
    max_week = df_x["week"].max()
    recent = df_x[df_x["week"] >= max_week - 3]

    # ② 商品の人気度（直近1ヶ月）
    item_popularity = (
        recent.groupby("article_id")["customer_id"]
        .count()
        .reset_index()
        .rename(columns={"customer_id": "item_popularity"})
    )

    # ③ ユーザーがそのカテゴリを買った回数
    user_category_count = (
        recent.merge(articles[["article_id", "index_group_name"]], on="article_id", how="left")
        .groupby(["customer_id", "index_group_name"])
        .size()
        .reset_index(name="user_category_count")
    )

    # ④ ユーザーの最終購買からの経過週数
    user_last_purchase = (
        df_x.groupby("customer_id")["week"]
        .max()
        .reset_index()
        .rename(columns={"week": "last_purchase_week"})
    )
    user_last_purchase["weeks_since_last_purchase"] = max_week - user_last_purchase["last_purchase_week"]

    # 特徴量を結合
    features = candidates.copy()
    features = features.merge(item_popularity, on="article_id", how="left")
    features["item_popularity"] = features["item_popularity"].fillna(0)
    features = features.merge(articles[["article_id", "index_group_name"]], on="article_id", how="left")
    features = features.merge(user_category_count, on=["customer_id", "index_group_name"], how="left")
    features["user_category_count"] = features["user_category_count"].fillna(0)
    features = features.merge(user_last_purchase[["customer_id", "weeks_since_last_purchase"]], on="customer_id", how="left")
    features["weeks_since_last_purchase"] = features["weeks_since_last_purchase"].fillna(999)

    return features

# 各foldで特徴量作成
for fold, target_week in enumerate([103, 102, 101], 1):
    print(f"\nfold{fold}の特徴量作成中...")
    train_x_fold = pd.read_csv(DATA_DIR / f'processed/train_x_fold{fold}.csv', dtype=dtype_config)
    candidates = pd.read_csv(DATA_DIR / f'processed/candidates_fold{fold}.csv', dtype={"article_id": "string"})
    
    features = create_features(candidates, train_x_fold, articles)
    features.to_csv(DATA_DIR / f'processed/features_fold{fold}.csv', index=False)
    print(f"fold{fold}: {len(features):,}行, カラム: {features.columns.tolist()}")


fold1の特徴量作成中...
fold1: 16,214,136行, カラム: ['customer_id', 'article_id', 'rank', 'item_popularity', 'index_group_name', 'user_category_count', 'weeks_since_last_purchase']

fold2の特徴量作成中...
fold2: 16,146,179行, カラム: ['customer_id', 'article_id', 'rank', 'item_popularity', 'index_group_name', 'user_category_count', 'weeks_since_last_purchase']

fold3の特徴量作成中...
fold3: 16,079,268行, カラム: ['customer_id', 'article_id', 'rank', 'item_popularity', 'index_group_name', 'user_category_count', 'weeks_since_last_purchase']


In [36]:
# 各foldで正解ラベルを付与
for fold, target_week in enumerate([103, 102, 101], 1):
    print(f"\nfold{fold}のラベル付与中...")
    features = pd.read_csv(DATA_DIR / f'processed/features_fold{fold}.csv', dtype={"article_id": "string"})
    train_y_fold = pd.read_csv(DATA_DIR / f'processed/train_y_fold{fold}.csv', dtype=dtype_config)
    
    # 正解ラベルを付与（買った=1、買わなかった=0）
    train_y_fold["label"] = 1
    features = features.merge(
        train_y_fold[["customer_id", "article_id", "label"]].drop_duplicates(),
        on=["customer_id", "article_id"],
        how="left"
    )
    features["label"] = features["label"].fillna(0).astype(int)
    
    features.to_csv(DATA_DIR / f'processed/features_fold{fold}.csv', index=False)
    
    pos = features["label"].sum()
    neg = len(features) - pos
    print(f"fold{fold}: 正例={pos:,} 負例={neg:,} 正例率={pos/len(features):.4f}")


fold1のラベル付与中...
fold1: 正例=7,283 負例=16,206,853 正例率=0.0004

fold2のラベル付与中...
fold2: 正例=7,838 負例=16,138,341 正例率=0.0005

fold3のラベル付与中...
fold3: 正例=6,704 負例=16,072,564 正例率=0.0004


In [37]:
for fold, target_week in enumerate([103, 102, 101], 1):
    features = pd.read_csv(DATA_DIR / f'processed/features_fold{fold}.csv', dtype={"article_id": "string"})
    train_y_fold = pd.read_csv(DATA_DIR / f'processed/train_y_fold{fold}.csv', dtype=dtype_config)
    
    recall = features["label"].sum() / len(train_y_fold)
    print(f"fold{fold}: train_y正解数={len(train_y_fold):,} 候補に含まれた正解数={features['label'].sum():,} Recall={recall:.4f}")

fold1: train_y正解数=240,643 候補に含まれた正解数=7,283 Recall=0.0303
fold2: train_y正解数=273,561 候補に含まれた正解数=7,838 Recall=0.0287
fold3: train_y正解数=269,876 候補に含まれた正解数=6,704 Recall=0.0248


In [38]:
# 各foldで候補生成（50件）
for fold, target_week in enumerate([103, 102, 101], 1):
    print(f"\nfold{fold}の候補生成中...")
    train_x_fold = pd.read_csv(DATA_DIR / f'processed/train_x_fold{fold}.csv', dtype=dtype_config)
    candidates = generate_candidates(train_x_fold, n_candidates=50)
    candidates.to_csv(DATA_DIR / f'processed/candidates_fold{fold}.csv', index=False)
    print(f"fold{fold}: 候補ペア数 {len(candidates):,}")


fold1の候補生成中...
fold1: 候補ペア数 67,553,447

fold2の候補生成中...
fold2: 候補ペア数 67,267,375

fold3の候補生成中...
fold3: 候補ペア数 66,987,257


In [39]:
# 各foldで特徴量作成
for fold, target_week in enumerate([103, 102, 101], 1):
    print(f"\nfold{fold}の特徴量作成中...")
    train_x_fold = pd.read_csv(DATA_DIR / f'processed/train_x_fold{fold}.csv', dtype=dtype_config)
    candidates = pd.read_csv(DATA_DIR / f'processed/candidates_fold{fold}.csv', dtype={"article_id": "string"})
    
    features = create_features(candidates, train_x_fold, articles)
    
    # 正解ラベルを付与
    train_y_fold = pd.read_csv(DATA_DIR / f'processed/train_y_fold{fold}.csv', dtype=dtype_config)
    train_y_fold["label"] = 1
    features = features.merge(
        train_y_fold[["customer_id", "article_id", "label"]].drop_duplicates(),
        on=["customer_id", "article_id"],
        how="left"
    )
    features["label"] = features["label"].fillna(0).astype(int)
    
    features.to_csv(DATA_DIR / f'processed/features_fold{fold}.csv', index=False)
    
    pos = features["label"].sum()
    neg = len(features) - pos
    recall = pos / len(train_y_fold)
    print(f"fold{fold}: 正例={pos:,} 負例={neg:,} 正例率={pos/len(features):.4f} Recall={recall:.4f}")


fold1の特徴量作成中...
fold1: 正例=14,328 負例=67,539,119 正例率=0.0002 Recall=0.0595

fold2の特徴量作成中...
fold2: 正例=16,732 負例=67,250,643 正例率=0.0002 Recall=0.0612

fold3の特徴量作成中...
fold3: 正例=12,353 負例=66,974,904 正例率=0.0002 Recall=0.0458


In [40]:
# 各foldで負例をダウンサンプリング
for fold in range(1, 4):
    print(f"\nfold{fold}のダウンサンプリング中...")
    features = pd.read_csv(DATA_DIR / f'processed/features_fold{fold}.csv', dtype={"article_id": "string"})
    
    pos = features[features["label"] == 1]
    neg = features[features["label"] == 0]
    
    # 負例を正例の10倍に絞る
    neg_sampled = neg.sample(n=len(pos) * 10, random_state=42)
    
    features_sampled = pd.concat([pos, neg_sampled], ignore_index=True).sample(frac=1, random_state=42)
    
    features_sampled.to_csv(DATA_DIR / f'processed/features_sampled_fold{fold}.csv', index=False)
    
    print(f"fold{fold}: 正例={len(pos):,} 負例={len(neg_sampled):,} 合計={len(features_sampled):,}")


fold1のダウンサンプリング中...
fold1: 正例=14,328 負例=143,280 合計=157,608

fold2のダウンサンプリング中...
fold2: 正例=16,732 負例=167,320 合計=184,052

fold3のダウンサンプリング中...
fold3: 正例=12,353 負例=123,530 合計=135,883


In [41]:
import lightgbm as lgb
import numpy as np

feature_cols = ["rank", "item_popularity", "user_category_count", "weeks_since_last_purchase"]

models = []

for fold in range(1, 4):
    print(f"\nfold{fold}の学習中...")
    features = pd.read_csv(DATA_DIR / f'processed/features_sampled_fold{fold}.csv', dtype={"article_id": "string"})
    
    X = features[feature_cols]
    y = features["label"]
    
    # LambdaRankはグループ情報が必要（ユーザーごとの候補数）
    group = features.groupby("customer_id").size().values
    
    train_data = lgb.Dataset(X, label=y, group=group)
    
    params = {
        "objective": "lambdarank",
        "metric": "ndcg",
        "ndcg_eval_at": [12],
        "learning_rate": 0.05,
        "num_leaves": 31,
        "min_data_in_leaf": 20,
        "verbose": -1,
    }
    
    model = lgb.train(
        params,
        train_data,
        num_boost_round=100,
    )
    
    models.append(model)
    print(f"fold{fold}: 学習完了")

print("\n全fold学習完了")



fold1の学習中...
fold1: 学習完了

fold2の学習中...
fold2: 学習完了

fold3の学習中...
fold3: 学習完了

全fold学習完了


In [47]:
# validデータの特徴量作成
print("validデータの特徴量作成中...")
valid_candidates = pd.read_csv(DATA_DIR / 'processed/candidates_fold1.csv', dtype={"article_id": "string"})

# valid_xで特徴量作成
valid_features = create_features(final_candidates, valid_x, articles)

print(f"valid特徴量: {len(valid_features):,}行")
print(valid_features.columns.tolist())

validデータの特徴量作成中...
valid特徴量: 16,276,619行
['customer_id', 'article_id', 'rank', 'item_popularity', 'index_group_name', 'user_category_count', 'weeks_since_last_purchase']


In [48]:
# 各foldのモデルで予測してアンサンブル
valid_features["score"] = 0
X_valid = valid_features[feature_cols]

for fold, model in enumerate(models, 1):
    valid_features["score"] += model.predict(X_valid) / len(models)

# ユーザーごとにスコア上位12件を選択
submission_lgb = (
    valid_features.sort_values(["customer_id", "score"], ascending=[True, False])
    .groupby("customer_id")
    .head(12)
    .reset_index(drop=True)
)

# MAP@12を計算
submission_lgb["rank"] = submission_lgb.groupby("customer_id").cumcount() + 1
map12_lgb = map_at_12(submission_lgb, valid_y)

print(f"\n【LightGBM LambdaRank】")
print(f"MAP@12: {map12_lgb:.4f}")
print(f"\n【ベースライン】")
print(f"MAP@12: {map12_baseline:.4f}")


【LightGBM LambdaRank】
MAP@12: 0.0000

【ベースライン】
MAP@12: 0.0091


In [49]:
print(valid_features["score"].describe())
print(valid_features[["customer_id", "article_id", "score", "rank"]].head(20))

count    1.627662e+07
mean    -3.892408e-01
std      1.340820e+00
min     -2.816904e+00
25%     -1.586366e+00
50%     -2.865245e-01
75%      5.558202e-01
max      4.135333e+00
Name: score, dtype: float64
    customer_id  article_id     score  rank
0             1  0568601043  2.392542     1
1             1  0751471001  0.871266     2
2             1  0706016001  0.700233     3
3             1  0915526001  0.624847     4
4             1  0918292001  0.669288     5
5             1  0751471043  0.582948     6
6             1  0915529003  0.989298     7
7             1  0896152002  1.132886     8
8             1  0898694001  1.213757     9
9             1  0448509014  1.040363    10
10            1  0863595006  0.967325    11
11            1  0916468003  1.028145    12
12            2  0751471001  0.248823     1
13            2  0706016001  0.552368     2
14            2  0915526001  0.094840     3
15            2  0918292001  0.026460     4
16            2  0751471043  0.051589     5
17  

In [51]:
print(submission_lgb[["customer_id", "article_id", "score", "rank"]].head(20))
print(f"\nrank分布: {submission_lgb['rank'].value_counts().sort_index()}")

    customer_id  article_id     score  rank
0             1  0568601043  2.392542     1
1             1  0898694001  1.213757     2
2             1  0896152002  1.132886     3
3             1  0448509014  1.040363     4
4             1  0916468003  1.028145     5
5             1  0915529003  0.989298     6
6             1  0863595006  0.967325     7
7             1  0751471001  0.871266     8
8             1  0706016001  0.700233     9
9             1  0918292001  0.669288    10
10            1  0915526001  0.624847    11
11            1  0751471043  0.582948    12
12            2  0706016001  0.552368     1
13            2  0896152002  0.433315     2
14            2  0915529003  0.414478     3
15            2  0915526002  0.381346     4
16            2  0448509014  0.337791     5
17            2  0898694001  0.332176     6
18            2  0751471001  0.248823     7
19            2  0916468003  0.221596     8

rank分布: rank
1     1357359
2     1357359
3     1357359
4     1357359
5     

In [52]:
# valid_yのarticle_idの型確認
print(f"valid_y article_id型: {valid_y['article_id'].dtype}")
print(valid_y["article_id"].head(3))

valid_y article_id型: int64
0    903762001
1    879189005
2    456163060
Name: article_id, dtype: int64


In [53]:
valid_y["article_id"] = valid_y["article_id"].astype("string").str.zfill(10)

print(valid_y["article_id"].head(3))

0    0903762001
1    0879189005
2    0456163060
Name: article_id, dtype: string


In [54]:
map12_lgb = map_at_12(submission_lgb, valid_y)
print(f"MAP@12: {map12_lgb:.4f}")

MAP@12: 0.0236


In [55]:
# 全データで候補生成
print("全データの候補生成中...")
transactions_all = pd.read_csv(DATA_DIR / 'processed/transactions_with_week.csv', dtype=dtype_config)
transactions_all["article_id"] = transactions_all["article_id"].astype("string").str.zfill(10)

candidates_all = generate_candidates(transactions_all, n_candidates=50)
candidates_all["article_id"] = candidates_all["article_id"].astype("string").str.zfill(10)

print(f"候補ペア数: {len(candidates_all):,}")

全データの候補生成中...
候補ペア数: 68,060,781


In [56]:
# 全データで特徴量作成
print("全データの特徴量作成中...")
candidates_all["article_id"] = candidates_all["article_id"].astype("string").str.zfill(10)
features_all = create_features(candidates_all, transactions_all, articles)
print(f"特徴量作成完了: {len(features_all):,}行")

全データの特徴量作成中...
特徴量作成完了: 68,060,781行


In [57]:
# 各foldのモデルでアンサンブル予測
print("予測中...")
X_all = features_all[feature_cols]
features_all["score"] = 0

for fold, model in enumerate(models, 1):
    features_all["score"] += model.predict(X_all) / len(models)

# ユーザーごとにスコア上位12件を選択
submission_all = (
    features_all.sort_values(["customer_id", "score"], ascending=[True, False])
    .groupby("customer_id")
    .head(12)
    .reset_index(drop=True)
)

submission_all["rank"] = submission_all.groupby("customer_id").cumcount() + 1

print(f"提出ユーザー数: {len(submission_all['customer_id'].unique()):,}")
print(submission_all.head(10))

予測中...
提出ユーザー数: 1,362,281
   customer_id  article_id  rank  item_popularity index_group_name  \
0            1  0568601043     1              372       Ladieswear   
1            1  0909370001     2             1868       Ladieswear   
2            1  0751471043     3             2132       Ladieswear   
3            1  0898694001     4             1989       Ladieswear   
4            1  0714790020     5             1529          Divided   
5            1  0850917001     6             1586       Ladieswear   
6            1  0706016001     7             2017          Divided   
7            1  0448509014     8             1844          Divided   
8            1  0918522001     9             1702       Ladieswear   
9            1  0915529003    10             2186       Ladieswear   

   user_category_count  weeks_since_last_purchase     score  
0                  1.0                          2  2.086546  
1                  1.0                          2  1.226386  
2                

In [58]:
# 全ユーザーを取得
customers = pd.read_csv(DATA_DIR / 'raw/customers.csv', usecols=['customer_id'])

# 提出形式に変換
submission_final = (
    submission_all.sort_values(["customer_id", "rank"])
    .groupby("customer_id")["article_id"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
    .rename(columns={"article_id": "prediction"})
)

# 候補生成に含まれなかったユーザーには人気上位12件を割り当て
top12_str = " ".join(
    transactions_all[transactions_all["week"] >= transactions_all["week"].max() - 3]
    .groupby("article_id")["customer_id"]
    .count()
    .sort_values(ascending=False)
    .head(12)
    .index.astype(str)
    .str.zfill(10)
    .tolist()
)

submission_full = customers.merge(submission_final, on="customer_id", how="left")
submission_full["prediction"] = submission_full["prediction"].fillna(top12_str)

# 保存
OUTPUT_DIR = (Path.cwd() / '../../outputs/submissions').resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
submission_full.to_csv(OUTPUT_DIR / 'submission_lgb_lambdarank.csv', index=False)

print(f"保存完了")
print(f"全ユーザー数: {len(submission_full):,}")
print(submission_full.head(5))

ValueError: You are trying to merge on str and int64 columns for key 'customer_id'. If you wish to proceed you should use pd.concat

In [60]:
print(f"customers customer_id型: {customers['customer_id'].dtype}")
print(f"submission_final customer_id型: {submission_final['customer_id'].dtype}")
OUTPUT_DIR = (Path.cwd() / '../../outputs/submissions').resolve()
# 型を統一
customers["customer_id"] = customers["customer_id"].astype("string")
submission_final["customer_id"] = submission_final["customer_id"].astype("string")

# 再度マージ
submission_full = customers.merge(submission_final, on="customer_id", how="left")
submission_full["prediction"] = submission_full["prediction"].fillna(top12_str)

# 保存
submission_full.to_csv(OUTPUT_DIR / 'submission_lgb_lambdarank.csv', index=False)

print(f"保存完了")
print(f"全ユーザー数: {len(submission_full):,}")
print(submission_full.head(5))

customers customer_id型: string
submission_final customer_id型: string
保存完了
全ユーザー数: 1,371,980
                                         customer_id  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...   
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   
3  00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...   
4  00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...   

                                          prediction  
0  0751471001 0915526001 0915529003 0751471043 09...  
1  0751471001 0915526001 0915529003 0751471043 09...  
2  0751471001 0915526001 0915529003 0751471043 09...  
3  0751471001 0915526001 0915529003 0751471043 09...  
4  0751471001 0915526001 0915529003 0751471043 09...  


In [4]:
import polars as pl
from pathlib import Path

DATA_DIR = (Path.cwd() / '../../data').resolve()

# データ読み込み
df = pl.read_csv(
    DATA_DIR / 'raw/transactions_train.csv',
    columns=['t_dat', 'customer_id', 'article_id'],
    schema_overrides={'article_id': pl.String}
)

df = df.with_columns(pl.col('t_dat').str.to_date())
last_ts = df['t_dat'].max()

print(f"最終日: {last_ts}")
print(f"行数: {len(df):,}")

最終日: 2020-09-22
行数: 31,788,324


In [1]:
import pandas as pd

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = (Path.cwd() / '../../data').resolve()

# データ読み込み
transactions = pd.read_csv(DATA_DIR / 'processed/transactions_with_week.csv',
                           usecols=['customer_id', 'article_id', 'week'],
                           dtype={'article_id': 'string', 'week': 'int8'})

customers = pd.read_csv(DATA_DIR / 'raw/customers.csv', usecols=['customer_id'])

print(f"transactions: {len(transactions):,}行")
print(f"customers: {len(customers):,}行")

transactions: 31,788,324行
customers: 1,371,980行


In [3]:
max_week = transactions['week'].max()

# 直近1ヶ月の購買履歴（新しい順にrank付与）
recent_purchase = (
    transactions[transactions['week'] >= max_week - 3]
    [['customer_id', 'article_id', 'week']]
    .drop_duplicates(subset=['customer_id', 'article_id'])
    .sort_values(['customer_id', 'week'], ascending=[True, False])
)
recent_purchase['rank'] = recent_purchase.groupby('customer_id').cumcount() + 1
recent_purchase = recent_purchase[recent_purchase['rank'] <= 12]

# 全体トレンド（直近1ヶ月の人気上位12件）
trend_top12 = (
    transactions[transactions['week'] >= max_week - 3]
    .groupby('article_id')['customer_id']
    .count()
    .sort_values(ascending=False)
    .head(12)
    .index.tolist()
)

print(f"リピート購買候補: {len(recent_purchase):,}行")
print(f"トレンド上位12件: {trend_top12}")

リピート購買候補: 835,527行
トレンド上位12件: ['0751471001', '0915526001', '0915529003', '0751471043', '0918292001', '0706016001', '0898694001', '0909370001', '0448509014', '0924243001', '0918522001', '0863595006']


In [6]:
# ユーザーごとの購買件数
user_purchase_count = (
    recent_purchase.groupby('customer_id')['rank'].max()
    .reset_index(name='purchase_count')
)

# 全ユーザーにマージ
all_users = customers['customer_id'].values
user_purchase_count = pd.DataFrame({'customer_id': all_users}).merge(
    user_purchase_count, on='customer_id', how='left'
)
user_purchase_count['purchase_count'] = user_purchase_count['purchase_count'].fillna(0).astype(int)

# トレンドで補完
trend_top12_str = [str(a).zfill(10) for a in trend_top12]

pop_candidates = pd.DataFrame({
    'customer_id': np.repeat(all_users, 12),
    'article_id': np.tile(trend_top12_str, len(all_users)),
    'pop_rank': np.tile(np.arange(1, 13), len(all_users))
})

# 購買済み商品を除外
purchased_set = recent_purchase[['customer_id', 'article_id']].copy()
purchased_set['article_id'] = purchased_set['article_id'].astype('string').str.zfill(10)
purchased_set['is_purchased'] = True

pop_candidates = pop_candidates.merge(purchased_set, on=['customer_id', 'article_id'], how='left')
pop_candidates = pop_candidates[pop_candidates['is_purchased'].isna()].drop(columns=['is_purchased'])

# rankを付与
pop_candidates = pop_candidates.merge(user_purchase_count, on='customer_id', how='left')
pop_candidates['rank'] = pop_candidates['pop_rank'] + pop_candidates['purchase_count']
pop_candidates = pop_candidates[pop_candidates['rank'] <= 12][['customer_id', 'article_id', 'rank']]

# 購買履歴のarticle_idを0埋め
recent_purchase['article_id'] = recent_purchase['article_id'].astype('string').str.zfill(10)

# 結合
final = pd.concat([
    recent_purchase[['customer_id', 'article_id', 'rank']],
    pop_candidates
], ignore_index=True).sort_values(['customer_id', 'rank'])

# 提出形式に変換
submission = (
    final.groupby('customer_id')['article_id']
    .apply(lambda x: ' '.join(x.astype(str)))
    .reset_index()
    .rename(columns={'article_id': 'prediction'})
)

# 購買履歴のないユーザーにはトレンド上位12件を割り当て
trend_str = ' '.join(trend_top12_str)
customers['customer_id'] = customers['customer_id'].astype('string')
submission['customer_id'] = submission['customer_id'].astype('string')
submission_full = customers.merge(submission, on='customer_id', how='left')
submission_full['prediction'] = submission_full['prediction'].fillna(trend_str)

# 保存
OUTPUT_DIR = (Path.cwd() / '../../outputs/submissions').resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
submission_full.to_csv(OUTPUT_DIR / 'submission_repeat_trend.csv', index=False)

print(f"保存完了")
print(f"全ユーザー数: {len(submission_full):,}")
print(submission_full.head(5))

保存完了
全ユーザー数: 1,371,980
                                         customer_id  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...   
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   
3  00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...   
4  00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...   

                                          prediction  
0  0751471001 0915526001 0915529003 0751471043 09...  
1  0751471001 0915526001 0915529003 0751471043 09...  
2  0751471001 0915526001 0915529003 0751471043 09...  
3  0751471001 0915526001 0915529003 0751471043 09...  
4  0751471001 0915526001 0915529003 0751471043 09...  


In [7]:
sub = pd.read_csv(OUTPUT_DIR / 'submission_repeat_trend.csv')
print(sub.head(10))
print(f"\n全ユーザー数: {len(sub):,}")

# predictionが全員同じになっていないか確認
print(f"\nユニークなprediction数: {sub['prediction'].nunique():,}")

# リピート購買が反映されているか確認
# 購買履歴があるユーザーのpredictionを確認
sample_user = recent_purchase['customer_id'].iloc[0]
print(f"\nサンプルユーザー: {sample_user}")
print(f"購買履歴: {recent_purchase[recent_purchase['customer_id'] == sample_user]['article_id'].tolist()}")
print(f"prediction: {sub[sub['customer_id'] == sample_user]['prediction'].values}")

                                         customer_id  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...   
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   
3  00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...   
4  00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...   
5  000064249685c11552da43ef22a5030f35a147f723d5b0...   
6  0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d...   
7  00007d2de826758b65a93dd24ce629ed66842531df6699...   
8  00007e8d4e54114b5b2a9b51586325a8d0fa74ea23ef77...   
9  00008469a21b50b3d147c97135e25b4201a8c58997f787...   

                                          prediction  
0  0751471001 0915526001 0915529003 0751471043 09...  
1  0751471001 0915526001 0915529003 0751471043 09...  
2  0751471001 0915526001 0915529003 0751471043 09...  
3  0751471001 0915526001 0915529003 0751471043 09...  
4  0751471001 0915526001 0915529003 0751471043 09...  
5  0751471001 0915526001 0915529003 0751471043 09... 

In [9]:
print("transactions customer_id:")
print(transactions['customer_id'].head(3))
print(f"型: {transactions['customer_id'].dtype}")

print("\ncustomers customer_id:")
print(customers['customer_id'].head(3))
print(f"型: {customers['customer_id'].dtype}")

transactions customer_id:
0    3
1    3
2    8
Name: customer_id, dtype: int64
型: int64

customers customer_id:
0    00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...
1    0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...
2    000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...
Name: customer_id, dtype: string
型: string


In [10]:
# customer_id_mapを読み込む
customer_id_map = pd.read_csv(DATA_DIR / 'processed/customer_id_map.csv',
                               dtype={'customer_id': 'string'})
print(customer_id_map.head(3))
print(customer_id_map.dtypes)

                                         customer_id  customer_id_int
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...                1
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...                2
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...                3
customer_id        string
customer_id_int     int64
dtype: object


In [13]:
recent_purchase['customer_id'] = recent_purchase['customer_id'].astype('int64')

recent_purchase = recent_purchase.merge(
    customer_id_map,
    left_on='customer_id',
    right_on='customer_id_int',
    how='left'
).drop(columns=['customer_id', 'customer_id_int']).rename(columns={'customer_id': 'customer_id_orig'})

recent_purchase = recent_purchase.rename(columns={'customer_id_orig': 'customer_id'})

print(recent_purchase.head(3))

KeyError: "['customer_id'] not found in axis"

In [14]:
recent_purchase['customer_id'] = recent_purchase['customer_id'].astype('int64')

temp = recent_purchase.merge(
    customer_id_map,
    left_on='customer_id',
    right_on='customer_id_int',
    how='left'
)

print(temp.columns.tolist())
print(temp.head(3))

['customer_id_x', 'article_id', 'week', 'rank', 'customer_id_y', 'customer_id_int']
   customer_id_x  article_id  week  rank  \
0              1  0568601043   102     1   
1              3  0794321007   103     1   
2              7  0719530003   103     1   

                                       customer_id_y  customer_id_int  
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...                1  
1  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...                3  
2  0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d...                7  


In [15]:
recent_purchase = temp.drop(columns=['customer_id_x', 'customer_id_int']).rename(columns={'customer_id_y': 'customer_id'})

print(recent_purchase.head(3))
print(f"型: {recent_purchase['customer_id'].dtype}")

   article_id  week  rank                                        customer_id
0  0568601043   102     1  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...
1  0794321007   103     1  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...
2  0719530003   103     1  0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d...
型: string


In [16]:
# ユーザーごとの購買件数
user_purchase_count = (
    recent_purchase.groupby('customer_id')['rank'].max()
    .reset_index(name='purchase_count')
)

# 全ユーザーにマージ
user_purchase_count = customers.merge(user_purchase_count, on='customer_id', how='left')
user_purchase_count['purchase_count'] = user_purchase_count['purchase_count'].fillna(0).astype(int)

# トレンド上位12件で補完
trend_top12_str = [str(a).zfill(10) for a in trend_top12]
all_users = customers['customer_id'].values

pop_candidates = pd.DataFrame({
    'customer_id': np.repeat(all_users, 12),
    'article_id': np.tile(trend_top12_str, len(all_users)),
    'pop_rank': np.tile(np.arange(1, 13), len(all_users))
})

# 購買済み商品を除外
purchased_set = recent_purchase[['customer_id', 'article_id']].copy()
purchased_set['is_purchased'] = True
pop_candidates = pop_candidates.merge(purchased_set, on=['customer_id', 'article_id'], how='left')
pop_candidates = pop_candidates[pop_candidates['is_purchased'].isna()].drop(columns=['is_purchased'])

# rankを付与
pop_candidates = pop_candidates.merge(user_purchase_count[['customer_id', 'purchase_count']], on='customer_id', how='left')
pop_candidates['rank'] = pop_candidates['pop_rank'] + pop_candidates['purchase_count']
pop_candidates = pop_candidates[pop_candidates['rank'] <= 12][['customer_id', 'article_id', 'rank']]

# 結合
final = pd.concat([
    recent_purchase[['customer_id', 'article_id', 'rank']],
    pop_candidates
], ignore_index=True).sort_values(['customer_id', 'rank'])

# 提出形式に変換
submission = (
    final.groupby('customer_id')['article_id']
    .apply(lambda x: ' '.join(x.astype(str)))
    .reset_index()
    .rename(columns={'article_id': 'prediction'})
)

# 購買履歴のないユーザーにはトレンド上位12件
trend_str = ' '.join(trend_top12_str)
submission_full = customers.merge(submission, on='customer_id', how='left')
submission_full['prediction'] = submission_full['prediction'].fillna(trend_str)

# 保存
OUTPUT_DIR = (Path.cwd() / '../../outputs/submissions').resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
submission_full.to_csv(OUTPUT_DIR / 'submission_repeat_trend.csv', index=False)

print(f"保存完了: {len(submission_full):,}ユーザー")
print(f"ユニークなprediction数: {submission_full['prediction'].nunique():,}")
print(submission_full.head(5))

保存完了: 1,371,980ユーザー
ユニークなprediction数: 175,108
                                         customer_id  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...   
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   
3  00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...   
4  00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...   

                                          prediction  
0  0568601043 0751471001 0915526001 0915529003 07...  
1  0751471001 0915526001 0915529003 0751471043 09...  
2  0794321007 0751471001 0915526001 0915529003 07...  
3  0751471001 0915526001 0915529003 0751471043 09...  
4  0751471001 0915526001 0915529003 0751471043 09...  


In [17]:
print(final.columns.tolist())
print(final.head(3))
print(f"行数: {len(final):,}")

['customer_id', 'article_id', 'rank']
                                              customer_id  article_id  rank
0       00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  0568601043     1
835527  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  0751471001     2
835528  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  0915526001     3
行数: 16,452,512


In [18]:
print(f"transactions: {'transactions' in dir() and len(transactions):,}")
print(f"articles: {'articles' in dir() and len(articles):,}")
print(f"customer_id_map: {'customer_id_map' in dir() and len(customer_id_map):,}")

transactions: 31,788,324
articles: 0
customer_id_map: 1,371,980


In [19]:
articles = pd.read_csv(DATA_DIR / 'raw/articles.csv', 
                       dtype={'article_id': 'string'})
print(f"articles: {len(articles):,}行")
print(articles.columns.tolist())

articles: 105,542行
['article_id', 'product_code', 'prod_name', 'product_type_no', 'product_type_name', 'product_group_name', 'graphical_appearance_no', 'graphical_appearance_name', 'colour_group_code', 'colour_group_name', 'perceived_colour_value_id', 'perceived_colour_value_name', 'perceived_colour_master_id', 'perceived_colour_master_name', 'department_no', 'department_name', 'index_code', 'index_name', 'index_group_no', 'index_group_name', 'section_no', 'section_name', 'garment_group_no', 'garment_group_name', 'detail_desc']


In [20]:
max_week = transactions['week'].max()
recent = transactions[transactions['week'] >= max_week - 3].copy()

# customer_idを文字列に変換
recent = recent.merge(customer_id_map, left_on='customer_id', right_on='customer_id_int', how='left')
recent = recent.drop(columns=['customer_id', 'customer_id_int']).rename(columns={'customer_id_y': 'customer_id'})
recent['article_id'] = recent['article_id'].astype('string').str.zfill(10)

# ① 候補生成時のrank（既にfinalにある）

# ② 商品の人気度（直近1ヶ月）
item_popularity = (
    recent.groupby('article_id')['customer_id']
    .count()
    .reset_index()
    .rename(columns={'customer_id': 'item_popularity'})
)

# ③ ユーザーがそのカテゴリを買った回数
user_category_count = (
    recent.merge(articles[['article_id', 'index_group_name']], on='article_id', how='left')
    .groupby(['customer_id', 'index_group_name'])
    .size()
    .reset_index(name='user_category_count')
)

# ④ ユーザーの最終購買からの経過週数
user_last_purchase = (
    transactions.groupby('customer_id')['week']
    .max()
    .reset_index()
    .rename(columns={'week': 'last_purchase_week'})
)
user_last_purchase = user_last_purchase.merge(customer_id_map, left_on='customer_id', right_on='customer_id_int', how='left')
user_last_purchase = user_last_purchase.drop(columns=['customer_id', 'customer_id_int']).rename(columns={'customer_id_y': 'customer_id'})
user_last_purchase['weeks_since_last_purchase'] = max_week - user_last_purchase['last_purchase_week']

# finalに特徴量を結合
features = final.copy()
features = features.merge(item_popularity, on='article_id', how='left')
features['item_popularity'] = features['item_popularity'].fillna(0)
features = features.merge(articles[['article_id', 'index_group_name']], on='article_id', how='left')
features = features.merge(user_category_count, on=['customer_id', 'index_group_name'], how='left')
features['user_category_count'] = features['user_category_count'].fillna(0)
features = features.merge(user_last_purchase[['customer_id', 'weeks_since_last_purchase']], on='customer_id', how='left')
features['weeks_since_last_purchase'] = features['weeks_since_last_purchase'].fillna(999)

print(features.columns.tolist())
print(f"行数: {len(features):,}")
print(features.head(3))

KeyError: "['customer_id'] not found in axis"

In [21]:
temp = transactions[transactions['week'] >= max_week - 3].copy()
temp = temp.merge(customer_id_map, left_on='customer_id', right_on='customer_id_int', how='left')
print(temp.columns.tolist())
print(temp.head(3))

['customer_id_x', 'article_id', 'week', 'customer_id_y', 'customer_id_int']
   customer_id_x  article_id  week  \
0             39  0572998007   101   
1            204  0808462002   101   
2            204  0868134001   101   

                                       customer_id_y  customer_id_int  
0  0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...               39  
1  0008968c0d451dbc5a9968da03196fe20051965edde741...              204  
2  0008968c0d451dbc5a9968da03196fe20051965edde741...              204  


In [22]:
recent = temp.drop(columns=['customer_id_x', 'customer_id_int']).rename(columns={'customer_id_y': 'customer_id'})
recent['article_id'] = recent['article_id'].astype('string').str.zfill(10)

print(recent.head(3))

   article_id  week                                        customer_id
0  0572998007   101  0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...
1  0808462002   101  0008968c0d451dbc5a9968da03196fe20051965edde741...
2  0868134001   101  0008968c0d451dbc5a9968da03196fe20051965edde741...


In [23]:
# ② 商品の人気度（直近1ヶ月）
item_popularity = (
    recent.groupby('article_id')['customer_id']
    .count()
    .reset_index()
    .rename(columns={'customer_id': 'item_popularity'})
)

# ③ ユーザーがそのカテゴリを買った回数
user_category_count = (
    recent.merge(articles[['article_id', 'index_group_name']], on='article_id', how='left')
    .groupby(['customer_id', 'index_group_name'])
    .size()
    .reset_index(name='user_category_count')
)

# ④ ユーザーの最終購買からの経過週数
user_last_purchase = (
    transactions.groupby('customer_id')['week']
    .max()
    .reset_index()
    .rename(columns={'week': 'last_purchase_week'})
)
user_last_purchase = user_last_purchase.merge(
    customer_id_map, left_on='customer_id', right_on='customer_id_int', how='left'
).drop(columns=['customer_id_x', 'customer_id_int']).rename(columns={'customer_id_y': 'customer_id'})
user_last_purchase['weeks_since_last_purchase'] = max_week - user_last_purchase['last_purchase_week']

# finalに特徴量を結合
features = final.copy()
features = features.merge(item_popularity, on='article_id', how='left')
features['item_popularity'] = features['item_popularity'].fillna(0)
features = features.merge(articles[['article_id', 'index_group_name']], on='article_id', how='left')
features = features.merge(user_category_count, on=['customer_id', 'index_group_name'], how='left')
features['user_category_count'] = features['user_category_count'].fillna(0)
features = features.merge(user_last_purchase[['customer_id', 'weeks_since_last_purchase']], on='customer_id', how='left')
features['weeks_since_last_purchase'] = features['weeks_since_last_purchase'].fillna(999)

print(features.columns.tolist())
print(f"行数: {len(features):,}")
print(features.head(3))

['customer_id', 'article_id', 'rank', 'item_popularity', 'index_group_name', 'user_category_count', 'weeks_since_last_purchase']
行数: 16,452,512
                                         customer_id  article_id  rank  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  0568601043     1   
1  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  0751471001     2   
2  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  0915526001     3   

   item_popularity index_group_name  user_category_count  \
0              372       Ladieswear                  1.0   
1             2589       Ladieswear                  1.0   
2             2249       Ladieswear                  1.0   

   weeks_since_last_purchase  
0                        2.0  
1                        2.0  
2                        2.0  


In [24]:
valid_y = pd.read_csv(DATA_DIR / 'processed/valid_y.csv',
                      dtype={'article_id': 'string'})
valid_y['article_id'] = valid_y['article_id'].astype('string').str.zfill(10)
valid_y = valid_y.merge(customer_id_map, left_on='customer_id', right_on='customer_id_int', how='left')
valid_y = valid_y.drop(columns=['customer_id_x', 'customer_id_int']).rename(columns={'customer_id_y': 'customer_id'})

# 正解ラベルを付与
valid_y['label'] = 1
features = features.merge(
    valid_y[['customer_id', 'article_id', 'label']].drop_duplicates(),
    on=['customer_id', 'article_id'],
    how='left'
)
features['label'] = features['label'].fillna(0).astype(int)

pos = features['label'].sum()
print(f"正例数: {pos:,}")
print(f"負例数: {len(features) - pos:,}")
print(f"Recall: {pos / len(valid_y):.4f}")

正例数: 184,952
負例数: 16,267,560
Recall: 0.8672


In [25]:
# train用データの準備（week103をtrain_y、それ以前をtrain_x）
train_y_week = 103
train_x = transactions[transactions['week'] < train_y_week].copy()
train_y_raw = transactions[transactions['week'] == train_y_week].copy()

# train用候補生成（直近1ヶ月のリピート購買）
max_week_train = train_x['week'].max()
recent_train = train_x[train_x['week'] >= max_week_train - 3].copy()

# customer_idを文字列に変換
recent_train = recent_train.merge(
    customer_id_map, left_on='customer_id', right_on='customer_id_int', how='left'
).drop(columns=['customer_id_x', 'customer_id_int']).rename(columns={'customer_id_y': 'customer_id'})
recent_train['article_id'] = recent_train['article_id'].astype('string').str.zfill(10)

# リピート購買候補
repeat_train = (
    recent_train[['customer_id', 'article_id', 'week']]
    .drop_duplicates(subset=['customer_id', 'article_id'])
    .sort_values(['customer_id', 'week'], ascending=[True, False])
)
repeat_train['rank'] = repeat_train.groupby('customer_id').cumcount() + 1
repeat_train = repeat_train[repeat_train['rank'] <= 12]

# トレンド上位12件
trend_train = (
    recent_train.groupby('article_id')['customer_id']
    .count()
    .sort_values(ascending=False)
    .head(12)
    .index.tolist()
)

# 全ユーザー
all_users_train = recent_train['customer_id'].unique()
user_purchase_count_train = (
    repeat_train.groupby('customer_id')['rank'].max()
    .reindex(all_users_train, fill_value=0)
    .reset_index(name='purchase_count')
)

# トレンドで補完
pop_train = pd.DataFrame({
    'customer_id': np.repeat(all_users_train, 12),
    'article_id': np.tile(trend_train, len(all_users_train)),
    'pop_rank': np.tile(np.arange(1, 13), len(all_users_train))
})
purchased_set_train = repeat_train[['customer_id', 'article_id']].copy()
purchased_set_train['is_purchased'] = True
pop_train = pop_train.merge(purchased_set_train, on=['customer_id', 'article_id'], how='left')
pop_train = pop_train[pop_train['is_purchased'].isna()].drop(columns=['is_purchased'])
pop_train = pop_train.merge(user_purchase_count_train, on='customer_id', how='left')
pop_train['rank'] = pop_train['pop_rank'] + pop_train['purchase_count']
pop_train = pop_train[pop_train['rank'] <= 12][['customer_id', 'article_id', 'rank']]

# 結合
final_train = pd.concat([
    repeat_train[['customer_id', 'article_id', 'rank']],
    pop_train
], ignore_index=True).sort_values(['customer_id', 'rank'])

print(f"train候補ペア数: {len(final_train):,}")

train候補ペア数: 2,781,096


In [26]:
# train用特徴量作成
item_popularity_train = (
    recent_train.groupby('article_id')['customer_id']
    .count()
    .reset_index()
    .rename(columns={'customer_id': 'item_popularity'})
)

user_category_count_train = (
    recent_train.merge(articles[['article_id', 'index_group_name']], on='article_id', how='left')
    .groupby(['customer_id', 'index_group_name'])
    .size()
    .reset_index(name='user_category_count')
)

user_last_purchase_train = (
    train_x.groupby('customer_id')['week']
    .max()
    .reset_index()
    .rename(columns={'week': 'last_purchase_week'})
)
user_last_purchase_train = user_last_purchase_train.merge(
    customer_id_map, left_on='customer_id', right_on='customer_id_int', how='left'
).drop(columns=['customer_id_x', 'customer_id_int']).rename(columns={'customer_id_y': 'customer_id'})
user_last_purchase_train['weeks_since_last_purchase'] = max_week_train - user_last_purchase_train['last_purchase_week']

# 特徴量結合
features_train = final_train.copy()
features_train = features_train.merge(item_popularity_train, on='article_id', how='left')
features_train['item_popularity'] = features_train['item_popularity'].fillna(0)
features_train = features_train.merge(articles[['article_id', 'index_group_name']], on='article_id', how='left')
features_train = features_train.merge(user_category_count_train, on=['customer_id', 'index_group_name'], how='left')
features_train['user_category_count'] = features_train['user_category_count'].fillna(0)
features_train = features_train.merge(user_last_purchase_train[['customer_id', 'weeks_since_last_purchase']], on='customer_id', how='left')
features_train['weeks_since_last_purchase'] = features_train['weeks_since_last_purchase'].fillna(999)

# ラベル付与
train_y_raw = train_y_raw.merge(
    customer_id_map, left_on='customer_id', right_on='customer_id_int', how='left'
).drop(columns=['customer_id_x', 'customer_id_int']).rename(columns={'customer_id_y': 'customer_id'})
train_y_raw['article_id'] = train_y_raw['article_id'].astype('string').str.zfill(10)
train_y_raw['label'] = 1

features_train = features_train.merge(
    train_y_raw[['customer_id', 'article_id', 'label']].drop_duplicates(),
    on=['customer_id', 'article_id'],
    how='left'
)
features_train['label'] = features_train['label'].fillna(0).astype(int)

pos = features_train['label'].sum()
recall = pos / len(train_y_raw)
print(f"正例数: {pos:,}")
print(f"負例数: {len(features_train) - pos:,}")
print(f"Recall: {recall:.4f}")

正例数: 5,552
負例数: 2,775,544
Recall: 0.0231


In [27]:
print(f"train_y ユーザー数: {train_y_raw['customer_id'].nunique():,}")
print(f"train候補 ユーザー数: {final_train['customer_id'].nunique():,}")

train_y ユーザー数: 68,613
train候補 ユーザー数: 232,706


In [28]:
train_y_users = set(train_y_raw['customer_id'].unique())
train_candidate_users = set(final_train['customer_id'].unique())

overlap = train_y_users & train_candidate_users
print(f"train_y ユーザー数: {len(train_y_users):,}")
print(f"train候補 ユーザー数: {len(train_candidate_users):,}")
print(f"重複ユーザー数: {len(overlap):,}")
print(f"train_yのうち候補に含まれる割合: {len(overlap)/len(train_y_users):.4f}")

train_y ユーザー数: 68,613
train候補 ユーザー数: 232,706
重複ユーザー数: 30,773
train_yのうち候補に含まれる割合: 0.4485


In [29]:
# train_yの全ユーザーにもトレンド候補を追加
train_y_users_list = list(train_y_users)

pop_train_y = pd.DataFrame({
    'customer_id': np.repeat(train_y_users_list, 12),
    'article_id': np.tile(trend_train, len(train_y_users_list)),
    'pop_rank': np.tile(np.arange(1, 13), len(train_y_users_list))
})

# 既存候補と重複除去して結合
pop_train_y['rank'] = pop_train_y['pop_rank']
pop_train_y = pop_train_y[['customer_id', 'article_id', 'rank']]

final_train = pd.concat([
    final_train,
    pop_train_y
], ignore_index=True).drop_duplicates(subset=['customer_id', 'article_id'])

print(f"train候補ペア数: {len(final_train):,}")
print(f"train候補ユーザー数: {final_train['customer_id'].nunique():,}")

train候補ペア数: 3,374,848
train候補ユーザー数: 270,546


In [30]:
# 特徴量結合
features_train = final_train.copy()
features_train = features_train.merge(item_popularity_train, on='article_id', how='left')
features_train['item_popularity'] = features_train['item_popularity'].fillna(0)
features_train = features_train.merge(articles[['article_id', 'index_group_name']], on='article_id', how='left')
features_train = features_train.merge(user_category_count_train, on=['customer_id', 'index_group_name'], how='left')
features_train['user_category_count'] = features_train['user_category_count'].fillna(0)
features_train = features_train.merge(user_last_purchase_train[['customer_id', 'weeks_since_last_purchase']], on='customer_id', how='left')
features_train['weeks_since_last_purchase'] = features_train['weeks_since_last_purchase'].fillna(999)

# ラベル付与
features_train = features_train.merge(
    train_y_raw[['customer_id', 'article_id', 'label']].drop_duplicates(),
    on=['customer_id', 'article_id'],
    how='left'
)
features_train['label'] = features_train['label'].fillna(0).astype(int)

pos = features_train['label'].sum()
recall = pos / len(train_y_raw)
print(f"正例数: {pos:,}")
print(f"負例数: {len(features_train) - pos:,}")
print(f"Recall: {recall:.4f}")

正例数: 8,445
負例数: 3,366,403
Recall: 0.0351


In [31]:
# 候補数を50件に増やして再生成
trend_train_50 = (
    recent_train.groupby('article_id')['customer_id']
    .count()
    .sort_values(ascending=False)
    .head(50)
    .index.tolist()
)

# train_yの全ユーザーにトレンド50件を追加
all_train_users = list(set(list(recent_train['customer_id'].unique()) + list(train_y_raw['customer_id'].unique())))

pop_train_50 = pd.DataFrame({
    'customer_id': np.repeat(all_train_users, 50),
    'article_id': np.tile(trend_train_50, len(all_train_users)),
    'rank': np.tile(np.arange(1, 51), len(all_train_users))
})

final_train = pd.concat([
    repeat_train[['customer_id', 'article_id', 'rank']],
    pop_train_50
], ignore_index=True).drop_duplicates(subset=['customer_id', 'article_id'])

print(f"train候補ペア数: {len(final_train):,}")
print(f"train候補ユーザー数: {final_train['customer_id'].nunique():,}")

train候補ペア数: 14,344,977
train候補ユーザー数: 270,546


In [32]:
# 特徴量結合
features_train = final_train.copy()
features_train = features_train.merge(item_popularity_train, on='article_id', how='left')
features_train['item_popularity'] = features_train['item_popularity'].fillna(0)
features_train = features_train.merge(articles[['article_id', 'index_group_name']], on='article_id', how='left')
features_train = features_train.merge(user_category_count_train, on=['customer_id', 'index_group_name'], how='left')
features_train['user_category_count'] = features_train['user_category_count'].fillna(0)
features_train = features_train.merge(user_last_purchase_train[['customer_id', 'weeks_since_last_purchase']], on='customer_id', how='left')
features_train['weeks_since_last_purchase'] = features_train['weeks_since_last_purchase'].fillna(999)

# ラベル付与
features_train = features_train.merge(
    train_y_raw[['customer_id', 'article_id', 'label']].drop_duplicates(),
    on=['customer_id', 'article_id'],
    how='left'
)
features_train['label'] = features_train['label'].fillna(0).astype(int)

pos = features_train['label'].sum()
recall = pos / len(train_y_raw)
print(f"正例数: {pos:,}")
print(f"負例数: {len(features_train) - pos:,}")
print(f"Recall: {recall:.4f}")

正例数: 15,293
負例数: 14,329,684
Recall: 0.0636


In [33]:
import lightgbm as lgb

feature_cols = ['rank']

# valid特徴量とラベル（featuresはすでにある）
X_valid = features[feature_cols]
y_valid = features['label']
group_valid = features.groupby('customer_id').size().values

# train特徴量とラベル
X_train = features_train[feature_cols]
y_train = features_train['label']
group_train = features_train.groupby('customer_id').size().values

train_data = lgb.Dataset(X_train, label=y_train, group=group_train)
valid_data = lgb.Dataset(X_valid, label=y_valid, group=group_valid)

params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [12],
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1,
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=100,
    valid_sets=[valid_data],
)

print("学習完了")

学習完了


In [34]:
# 予測
features['score'] = model.predict(features[feature_cols])

# ユーザーごとにスコア上位12件を選択
submission_rerank = (
    features.sort_values(['customer_id', 'score'], ascending=[True, False])
    .groupby('customer_id')
    .head(12)
    .reset_index(drop=True)
)

# 提出形式に変換
submission_rerank_final = (
    submission_rerank.groupby('customer_id')['article_id']
    .apply(lambda x: ' '.join(x.astype(str)))
    .reset_index()
    .rename(columns={'article_id': 'prediction'})
)

# 購買履歴のないユーザーにはトレンド上位12件
trend_str = ' '.join([str(a).zfill(10) for a in trend_top12])
customers['customer_id'] = customers['customer_id'].astype('string')
submission_rerank_final['customer_id'] = submission_rerank_final['customer_id'].astype('string')
submission_full_rerank = customers.merge(submission_rerank_final, on='customer_id', how='left')
submission_full_rerank['prediction'] = submission_full_rerank['prediction'].fillna(trend_str)

# 保存
OUTPUT_DIR = (Path.cwd() / '../../outputs/submissions').resolve()
submission_full_rerank.to_csv(OUTPUT_DIR / 'submission_rerank.csv', index=False)

print(f"保存完了: {len(submission_full_rerank):,}ユーザー")
print(submission_full_rerank.head(5))

保存完了: 1,371,980ユーザー
                                         customer_id  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...   
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   
3  00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...   
4  00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...   

                                          prediction  
0  0751471001 0915529003 0898694001 0918292001 09...  
1  0915526001 0751471043 0909370001 0706016001 09...  
2  0751471001 0915529003 0898694001 0918292001 09...  
3  0915526001 0751471043 0909370001 0706016001 09...  
4  0915526001 0751471043 0909370001 0706016001 09...  


In [35]:
# valid_yのarticle_idを0埋め
valid_y['article_id'] = valid_y['article_id'].astype('string').str.zfill(10)

# Recallを計算
hit = final.merge(valid_y[['customer_id', 'article_id']], on=['customer_id', 'article_id'], how='inner')
recall = len(hit) / len(valid_y)
print(f"正解数: {len(valid_y):,}")
print(f"候補に含まれた正解数: {len(hit):,}")
print(f"Recall: {recall:.4f}")

正解数: 213,283
候補に含まれた正解数: 207,464
Recall: 0.9727


In [36]:
# finalの候補がどこから来ているか確認
print(f"finalの候補数: {len(final):,}")
print(f"ユーザー数: {final['customer_id'].nunique():,}")
print(f"1ユーザーあたりの平均候補数: {len(final) / final['customer_id'].nunique():.1f}")

# rank分布
print(final['rank'].value_counts().sort_index())

finalの候補数: 16,452,512
ユーザー数: 1,371,980
1ユーザーあたりの平均候補数: 12.0
rank
1     1371980
2     1371816
3     1371410
4     1371285
5     1371147
6     1371038
7     1370868
8     1370659
9     1370492
10    1370607
11    1370617
12    1370593
Name: count, dtype: int64
